# 🐟 Fish Speech - Google Colab Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fishaudio/fish-speech/blob/main/fish_speech_colab.ipynb)

**A powerful Text-to-Speech model with voice cloning capabilities**

- 🎯 **Zero-shot & Few-shot TTS**: Clone voices with just 10-30 seconds of audio
- 🌍 **Multilingual**: Supports English, Chinese, Japanese, Korean, French, German, Arabic, Spanish
- 🚀 **Fast**: Optimized for real-time inference
- 🎨 **WebUI**: Easy-to-use Gradio interface
- 🔗 **Live Sharing**: Get a public link to share your TTS interface

---

## 📋 Instructions

1. **Set Runtime**: Go to `Runtime` → `Change runtime type` → Select `GPU` (T4 or better recommended)
2. **Run All Cells**: Click `Runtime` → `Run all` or run cells one by one
3. **Wait for Setup**: Initial setup takes 5-10 minutes (model download + loading)
4. **Use the Interface**: A public Gradio link will appear when ready
5. **Share**: The Gradio link can be shared with others for 72 hours

---

⚠️ **Requirements**: This notebook requires a GPU runtime. Free Colab provides limited GPU hours.

## 🔧 Step 1: Environment Setup

In [ ]:
# Check GPU availability
import torch
import subprocess
import os

print("🔍 Checking GPU availability...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU detected: {gpu_name}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
    
    if gpu_memory < 10:
        print("⚠️  Warning: Less than 10GB GPU memory. Consider using --half flag for memory optimization.")
else:
    print("❌ No GPU detected! Please change runtime to GPU.")
    print("Go to Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU required for Fish Speech")

# Set locale for proper text handling
import locale
try:
    locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')
except:
    print("Note: Could not set UTF-8 locale, but this shouldn't affect functionality")

## 📥 Step 2: Clone Repository

In [ ]:
import os
import subprocess

# Clone the repository if not already present
if not os.path.exists('/content/fish-speech'):
    print("📥 Cloning Fish Speech repository...")
    !git clone https://github.com/fishaudio/fish-speech.git /content/fish-speech
    print("✅ Repository cloned successfully!")
else:
    print("✅ Repository already exists!")

# Change to the repository directory
os.chdir('/content/fish-speech')
print(f"📁 Current directory: {os.getcwd()}")

## 📦 Step 3: Install Dependencies

In [ ]:
print("📦 Installing Fish Speech dependencies...")
print("⏳ This may take 3-5 minutes...")

# Install the package in development mode
!pip install -e . --quiet

# Install additional dependencies that might be needed
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 --quiet

print("✅ Dependencies installed successfully!")

## 🤗 Step 4: Download Models

In [ ]:
import os
from huggingface_hub import hf_hub_download
from tqdm import tqdm

print("🤗 Downloading OpenAudio S1-mini model from Hugging Face...")
print("⏳ This may take 5-10 minutes depending on your connection...")

# Create checkpoints directory
os.makedirs('checkpoints/openaudio-s1-mini', exist_ok=True)

# Download the model using huggingface-cli for better progress tracking
!huggingface-cli download fishaudio/openaudio-s1-mini --local-dir checkpoints/openaudio-s1-mini/

# Verify model files
required_files = ['model.pth', 'codec.pth', 'config.json', 'tokenizer.tiktoken']
missing_files = []

for file in required_files:
    file_path = f'checkpoints/openaudio-s1-mini/{file}'
    if os.path.exists(file_path):
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"✅ {file}: {size:.1f} MB")
    else:
        missing_files.append(file)
        print(f"❌ {file}: Missing")

if missing_files:
    raise RuntimeError(f"Missing required model files: {missing_files}")
else:
    print("\n🎉 All model files downloaded successfully!")

## 🧠 Step 5: Load Models and Setup Inference Engine

In [ ]:
import os
import torch
import pyrootutils
from pathlib import Path
from loguru import logger

# Setup root path
pyrootutils.setup_root(__file__, indicator=".project-root", pythonpath=True)

# Import Fish Speech modules
from fish_speech.inference_engine import TTSInferenceEngine
from fish_speech.models.dac.inference import load_model as load_decoder_model
from fish_speech.models.text2semantic.inference import launch_thread_safe_queue
from fish_speech.utils.schema import ServeTTSRequest
from tools.webui import build_app
from tools.webui.inference import get_inference_wrapper

# Make einx happy
os.environ["EINX_FILTER_TRACEBACK"] = "false"

print("🧠 Loading models...")
print("⏳ This may take 2-3 minutes...")

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
precision = torch.bfloat16  # Use bfloat16 for better compatibility
compile_model = False  # Disable compilation for Colab compatibility

llama_checkpoint_path = Path("checkpoints/openaudio-s1-mini")
decoder_checkpoint_path = Path("checkpoints/openaudio-s1-mini/codec.pth")
decoder_config_name = "modded_dac_vq"

print(f"📱 Device: {device}")
print(f"🔢 Precision: {precision}")

# Load LLAMA model
print("\n🦙 Loading LLAMA model...")
llama_queue = launch_thread_safe_queue(
    checkpoint_path=llama_checkpoint_path,
    device=device,
    precision=precision,
    compile=compile_model,
)
print("✅ LLAMA model loaded!")

# Load VQ-GAN decoder model
print("\n🎵 Loading VQ-GAN decoder model...")
decoder_model = load_decoder_model(
    config_name=decoder_config_name,
    checkpoint_path=decoder_checkpoint_path,
    device=device,
)
print("✅ VQ-GAN decoder loaded!")

# Create inference engine
print("\n⚙️ Creating inference engine...")
inference_engine = TTSInferenceEngine(
    llama_queue=llama_queue,
    decoder_model=decoder_model,
    compile=compile_model,
    precision=precision,
)
print("✅ Inference engine created!")

# Warm up with a test inference
print("\n🔥 Warming up models...")
try:
    list(
        inference_engine.inference(
            ServeTTSRequest(
                text="Hello world.",
                references=[],
                reference_id=None,
                max_new_tokens=1024,
                chunk_length=200,
                top_p=0.7,
                repetition_penalty=1.5,
                temperature=0.7,
                format="wav",
            )
        )
    )
    print("✅ Warm-up completed successfully!")
except Exception as e:
    print(f"⚠️ Warm-up warning: {e}")
    print("This might not affect functionality, continuing...")

print("\n🎉 All models loaded and ready!")

## 🌐 Step 6: Launch Gradio WebUI

In [ ]:
import gradio as gr
from tools.webui import build_app
from tools.webui.inference import get_inference_wrapper

print("🌐 Launching Gradio WebUI...")
print("⏳ Setting up interface...")

# Get the inference function with the immutable arguments
inference_fct = get_inference_wrapper(inference_engine)

# Build the Gradio app
app = build_app(inference_fct, theme="light")

print("\n🚀 Starting Gradio interface...")
print("📱 This will create a public link that you can share!")
print("⏰ The link will be valid for 72 hours.")
print("\n" + "="*50)
print("🎉 FISH SPEECH IS READY!")
print("="*50)

# Launch with public sharing enabled
app.launch(
    share=True,           # Create public link
    debug=False,          # Disable debug for cleaner output
    show_api=False,       # Hide API docs for cleaner interface
    server_name="0.0.0.0", # Allow external connections
    server_port=7860,     # Standard Gradio port
    inbrowser=True        # Try to open in browser
)

## 📖 How to Use Fish Speech

### 🎯 Basic Text-to-Speech
1. **Enter text** in the "Input Text" field
2. **Click "Generate"** to create speech with a random voice
3. **Listen** to the generated audio

### 🎭 Voice Cloning (Recommended)
1. **Upload reference audio** (10-30 seconds of clear speech)
2. **Enter reference text** (what the person says in the audio)
3. **Enter your target text** (what you want them to say)
4. **Click "Generate"** to clone the voice

### ⚙️ Advanced Parameters
- **Temperature** (0.1-1.0): Lower = more consistent, Higher = more creative
- **Top P** (0.1-1.0): Controls randomness in generation
- **Repetition Penalty** (1.0-2.0): Prevents repetitive speech
- **Max New Tokens**: Length of generated speech

### 🌍 Multilingual Support
Fish Speech automatically detects language. Supported languages:
- 🇺🇸 English
- 🇨🇳 Chinese (Simplified/Traditional)
- 🇯🇵 Japanese
- 🇰🇷 Korean
- 🇫🇷 French
- 🇩🇪 German
- 🇸🇦 Arabic
- 🇪🇸 Spanish

### 💡 Tips for Best Results
1. **Reference Audio**: Use clear, noise-free audio with consistent volume
2. **Reference Text**: Make sure it exactly matches what's spoken
3. **Target Text**: Keep sentences reasonable length (not too long)
4. **Voice Consistency**: Use reference audio from the same speaker/session
5. **Language Mixing**: You can mix languages in the same text

### 🔗 Sharing Your Interface
- The Gradio link above is **public** and can be shared with anyone
- Link is valid for **72 hours** from creation
- Multiple people can use it simultaneously
- **Note**: Be mindful of Colab's usage limits

---

## 🛠️ Troubleshooting

### Common Issues:

**"CUDA out of memory"**
- Restart runtime: `Runtime` → `Restart runtime`
- Use shorter text inputs
- Try reducing `max_new_tokens`

**"No audio generated"**
- Check that your text is not empty
- Try simpler text without special characters
- Ensure reference audio is properly uploaded

**"Model loading failed"**
- Restart runtime and run all cells again
- Check internet connection
- Verify GPU is enabled

**"Gradio link not working"**
- Links expire after 72 hours
- Re-run the Gradio launch cell
- Check if Colab session is still active

### Performance Tips:
- **First generation** is always slower (model warm-up)
- **Shorter texts** generate faster
- **Colab Pro** provides better GPU access
- **Keep session active** to avoid reloading models

---

## 📚 Additional Resources

- 🏠 **Official Website**: [fish.audio](https://fish.audio)
- 📖 **Documentation**: [docs.fish.audio](https://docs.fish.audio)
- 💻 **GitHub Repository**: [fishaudio/fish-speech](https://github.com/fishaudio/fish-speech)
- 🤗 **Hugging Face**: [fishaudio](https://huggingface.co/fishaudio)
- 🎮 **Discord Community**: Join for support and discussions

---

## ⚖️ License & Usage

Fish Speech is released under **CC BY-NC-SA 4.0 License**.

**Please use responsibly:**
- 🚫 Do not use for illegal activities
- 🚫 Do not impersonate others without consent
- 🚫 Do not create misleading or harmful content
- ✅ Respect local laws and regulations
- ✅ Give credit when using generated content

**We are not responsible for any misuse of this technology.**

---

*Enjoy creating amazing speech with Fish Speech! 🐟🎵*